# Crop and Fertilizer Recommendation System

This project recommends a suitable **crop** and **fertilizer** based on:

- Taluka
- Soil Color
- Nitrogen (N)
- Phosphorus (P)
- Potassium (K)
- Soil pH
- Rainfall
- Temperature

The system uses a **Random Forest Classifier** to predict the recommended crop.

It also provides a **YouTube learning link** related to the recommended crop and fertilizer.

## Import Libraries

In [1]:
import pandas as pd
import numpy as np
import ipywidgets as widgets

from IPython.display import display, HTML
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

## Load Dataset

### Dataset

The dataset contains agricultural information for different regions/talukas, including soil properties, weather conditions, crops, and fertilizers.

In [3]:
dataset = pd.read_csv("FinalAmravati Crop and fertilizer dataset.csv")

dataset.head()

,Taluka_Name,Soil_color,Nitrogen,Phosphorus,Potassium,pH,Rainfall,Temperature,Crop,Fertilizer,Link
0,Amravati,Black,75,50,100,6.5,1000,20,Orange,Urea,https://youtu.be/2t5Am0xLTOo
1,Amravati,Black,80,50,100,6.5,1000,20,Orange,Urea,https://youtu.be/2t5Am0xLTOo
2,Amravati,Black,85,50,100,6.5,1000,20,Orange,Urea,https://youtu.be/2t5Am0xLTOo
3,Amravati,Black,90,50,100,6.5,1000,20,Orange,Urea,https://youtu.be/2t5Am0xLTOo
4,Amravati,Black,95,50,100,6.5,1000,20,Orange,Urea,https://youtu.be/2t5Am0xLTOo


## Create Widgets

In [4]:
taluka_widget = widgets.Combobox(
    placeholder='Choose Taluka',
    options=tuple(dataset['Taluka_Name'].dropna().unique()),
    description='Select Taluka:',
    ensure_option=True,
    disabled=False
)

soil_color_widget = widgets.Combobox(
    placeholder='Select Soil Color',
    options=tuple(dataset['Soil_color'].dropna().unique()),
    description='Soil Color:',
    ensure_option=True
)

nitrogen_widget = widgets.Combobox(
    description="Nitrogen:",
    placeholder='Select Nitrogen',
    ensure_option=True
)

phosphorus_widget = widgets.Combobox(
    description="Phosphorus:",
    placeholder='Select Phosphorus',
    ensure_option=True
)

potassium_widget = widgets.Combobox(
    description="Potassium:",
    placeholder='Select Potassium',
    ensure_option=True
)

ph_widget = widgets.Combobox(
    description="pH:",
    placeholder='Select pH',
    ensure_option=True
)

rainfall_widget = widgets.Combobox(
    description="Rainfall:",
    placeholder='Select Rainfall',
    ensure_option=True
)

temperature_widget = widgets.Combobox(
    description="Temperature:",
    placeholder='Select Temperature',
    ensure_option=True
)

recommend_widget = widgets.Output()

## Update Soil Color

In [5]:
def update_soil_color_options(change):
    taluka = change.new

    if taluka:
        soil_colors = dataset[
            dataset['Taluka_Name'] == taluka
        ]['Soil_color'].dropna().unique()

        soil_color_widget.options = tuple(soil_colors)

    else:
        soil_color_widget.options = ()

## Update Numeric Options

In [6]:
def get_filtered_data():
    taluka = taluka_widget.value
    soil_color = soil_color_widget.value

    if taluka and soil_color:
        return dataset[
            (dataset['Taluka_Name'] == taluka) &
            (dataset['Soil_color'] == soil_color)
        ]

    return pd.DataFrame()


def update_nitrogen_options(change=None):
    filtered = get_filtered_data()

    if not filtered.empty:
        nitrogen_widget.options = tuple(
            str(x) for x in filtered['Nitrogen'].dropna().unique()
        )
    else:
        nitrogen_widget.options = ()


def update_phosphorus_options(change=None):
    filtered = get_filtered_data()

    if not filtered.empty:
        phosphorus_widget.options = tuple(
            str(x) for x in filtered['Phosphorus'].dropna().unique()
        )
    else:
        phosphorus_widget.options = ()


def update_potassium_options(change=None):
    filtered = get_filtered_data()

    if not filtered.empty:
        potassium_widget.options = tuple(
            str(x) for x in filtered['Potassium'].dropna().unique()
        )
    else:
        potassium_widget.options = ()


def update_ph_options(change=None):
    filtered = get_filtered_data()

    if not filtered.empty:
        ph_widget.options = tuple(
            str(x) for x in filtered['pH'].dropna().unique()
        )
    else:
        ph_widget.options = ()


def update_rainfall_options(change=None):
    filtered = get_filtered_data()

    if not filtered.empty:
        rainfall_widget.options = tuple(
            str(x) for x in filtered['Rainfall'].dropna().unique()
        )
    else:
        rainfall_widget.options = ()


def update_temperature_options(change=None):
    filtered = get_filtered_data()

    if not filtered.empty:
        temperature_widget.options = tuple(
            str(x) for x in filtered['Temperature'].dropna().unique()
        )
    else:
        temperature_widget.options = ()

## YouTube Link Generator

In [7]:
from urllib.parse import quote_plus

def generate_youtube_link(crop, fertilizer):
    search_query = f"{crop} farming cultivation fertilizer {fertilizer}"
    
    youtube_url = (
        "https://www.youtube.com/results?search_query="
        + quote_plus(search_query)
    )
    
    return youtube_url

## Train Model and Recommend

In [ ]:
def train_model(change=None):

    with recommend_widget:
        recommend_widget.clear_output()

        try:
            # Get selected values
            taluka = taluka_widget.value
            soil_color = soil_color_widget.value

            nitrogen = float(nitrogen_widget.value)
            phosphorus = float(phosphorus_widget.value)
            potassium = float(potassium_widget.value)
            pH = float(ph_widget.value)
            rainfall = float(rainfall_widget.value)
            temperature = float(temperature_widget.value)

            # Validate selections
            if not taluka or not soil_color:
                print("Please select Taluka and Soil Color.")
                return

            # Prepare features
            categorical_columns = [
                'Taluka_Name',
                'Soil_color'
            ]

            numerical_columns = [
                'Nitrogen',
                'Phosphorus',
                'Potassium',
                'pH',
                'Rainfall',
                'Temperature'
            ]

            # One-hot encode categorical columns
            encoder = OneHotEncoder(
                handle_unknown='ignore',
                sparse_output=False
            )

            categorical_encoded = encoder.fit_transform(
                dataset[categorical_columns]
            )

            # Numerical features
            numerical_data = dataset[numerical_columns].astype(float).values

            # Combine numerical + categorical features
            X = np.hstack([
                numerical_data,
                categorical_encoded
            ])

            y = dataset['Crop']

            # Train-test split
            X_train, X_test, y_train, y_test = train_test_split(
                X,
                y,
                test_size=0.2,
                random_state=42,
                stratify=y
            )

            # Random Forest
            model_crop = RandomForestClassifier(
                n_estimators=100,
                random_state=42
            )

            model_crop.fit(X_train, y_train)

            # Prepare user input
            input_categorical = encoder.transform(
                pd.DataFrame({
                    'Taluka_Name': [taluka],
                    'Soil_color': [soil_color]
                })
            )

            input_numerical = np.array([[
                nitrogen,
                phosphorus,
                potassium,
                pH,
                rainfall,
                temperature
            ]])

            input_encoded = np.hstack([
                input_numerical,
                input_categorical
            ])

            # Prediction
            predicted_crop = model_crop.predict(input_encoded)[0]

            # Find fertilizer
            fertilizer_data = dataset[
                dataset['Crop'] == predicted_crop
            ]

            if not fertilizer_data.empty:
                recommended_fertilizer = fertilizer_data[
                    'Fertilizer'
                ].iloc[0]
            else:
                recommended_fertilizer = "Not available"

            # Generate YouTube link
            youtube_link = generate_youtube_link(
                predicted_crop,
                recommended_fertilizer
            )

            # Accuracy
            accuracy = model_crop.score(X_test, y_test)

            # Display result
            print("🌾 CROP & FERTILIZER RECOMMENDATION")
            print("=" * 45)

            print(f"📍 Taluka: {taluka}")
            print(f"🌱 Soil Color: {soil_color}")
            print(f"🧪 Nitrogen: {nitrogen}")
            print(f"🧪 Phosphorus: {phosphorus}")
            print(f"🧪 Potassium: {potassium}")
            print(f"⚗️ pH: {pH}")
            print(f"🌧️ Rainfall: {rainfall}")
            print(f"🌡️ Temperature: {temperature}")

            print("\n" + "=" * 45)

            print(f"🌾 Recommended Crop: {predicted_crop}")
            print(f"🧪 Recommended Fertilizer: {recommended_fertilizer}")

            print(f"\n📊 Model Accuracy: {accuracy * 100:.2f}%")

            display(
                HTML(
                    f'''
                    <h3>🎥 Learn More on YouTube</h3>
                    <a href="{youtube_link}" target="_blank">
                        ▶️ Watch {predicted_crop} Farming & Fertilizer Videos
                    </a>
                    '''
                )
            )

        except ValueError:
            print(" Please select valid numeric values.")

        except Exception as e:
            print(" Error:", e)

## Connecting Observers

In [9]:
taluka_widget.observe(
    update_soil_color_options,
    names='value'
)

taluka_widget.observe(
    update_nitrogen_options,
    names='value'
)

soil_color_widget.observe(
    update_nitrogen_options,
    names='value'
)

taluka_widget.observe(
    update_phosphorus_options,
    names='value'
)

soil_color_widget.observe(
    update_phosphorus_options,
    names='value'
)

taluka_widget.observe(
    update_potassium_options,
    names='value'
)

soil_color_widget.observe(
    update_potassium_options,
    names='value'
)

taluka_widget.observe(
    update_ph_options,
    names='value'
)

soil_color_widget.observe(
    update_ph_options,
    names='value'
)

taluka_widget.observe(
    update_rainfall_options,
    names='value'
)

soil_color_widget.observe(
    update_rainfall_options,
    names='value'
)

taluka_widget.observe(
    update_temperature_options,
    names='value'
)

soil_color_widget.observe(
    update_temperature_options,
    names='value'
)

## Display UI

In [10]:
button = widgets.Button(
    description="🌾 Recommend Crop and Fertilizer",
    button_style='success',
    tooltip='Generate recommendation'
)

button.on_click(train_model)

display(
    widgets.VBox([
        taluka_widget,
        soil_color_widget,
        nitrogen_widget,
        phosphorus_widget,
        potassium_widget,
        ph_widget,
        rainfall_widget,
        temperature_widget,
        button,
        recommend_widget
    ])
)